In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

y = df["converted"]
X = df.drop(columns=["converted"]).copy()

print("QUESTION 1")
print("Industry mode:", df["industry"].mode().iloc[0])

categorical = X.select_dtypes(include=["object"]).columns.tolist()
numerical = X.select_dtypes(exclude=["object"]).columns.tolist()

X[categorical] = X[categorical].fillna("NA")
X[numerical] = X[numerical].fillna(0.0)

print("\nQUESTION 2")
corr = X[numerical].corr().abs()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

for a, b in pairs:
    print(f"{a} and {b}: {corr.loc[a, b]:.4f}")

best_pair = max(pairs, key=lambda p: corr.loc[p[0], p[1]])
print("Biggest correlation:", best_pair)

data = X.copy()
data["converted"] = y

df_full_train, df_test = train_test_split(
    data, test_size=0.2, random_state=42
)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train.pop("converted")
y_val = df_val.pop("converted")

print("\nQUESTION 3")
mi = {
    col: mutual_info_score(y_train, df_train[col])
    for col in categorical
}
for col, score in sorted(mi.items(), key=lambda item: -item[1]):
    print(f"{col}: {round(score, 2)}")

print("Biggest MI:", max(mi, key=mi.get))

def train_model(train_X, val_X, y_train, C=1.0):
    dv = DictVectorizer(sparse=False)

    train_matrix = dv.fit_transform(
        train_X.to_dict(orient="records")
    )
    val_matrix = dv.transform(
        val_X.to_dict(orient="records")
    )

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    )
    model.fit(train_matrix, y_train)

    predictions = model.predict(val_matrix)
    return accuracy_score, predictions

print("\nQUESTION 4")
dv = DictVectorizer(sparse=False)
X_train_matrix = dv.fit_transform(
    df_train.to_dict(orient="records")
)
X_val_matrix = dv.transform(
    df_val.to_dict(orient="records")
)

model = LogisticRegression(
    solver="liblinear", C=1.0,
    max_iter=1000, random_state=42
)
model.fit(X_train_matrix, y_train)
predictions = model.predict(X_val_matrix)

original_accuracy = accuracy_score(y_val, predictions)
print("Validation accuracy:", round(original_accuracy, 2))

print("\nQUESTION 5")
features_to_test = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]
differences = {}
for col in features_to_test:
    reduced_train = df_train.drop(columns=[col])
    reduced_val = df_val.drop(columns=[col])

    dv_reduced = DictVectorizer(sparse=False)
    train_matrix = dv_reduced.fit_transform(
        reduced_train.to_dict(orient="records")
    )
    val_matrix = dv_reduced.transform(
        reduced_val.to_dict(orient="records")
    )

    reduced_model = LogisticRegression(
        solver="liblinear", C=1.0,
        max_iter=1000, random_state=42
    )
    reduced_model.fit(train_matrix, y_train)
    reduced_predictions = reduced_model.predict(val_matrix)

    reduced_accuracy = accuracy_score(y_val, reduced_predictions)
    differences[col] = abs(original_accuracy - reduced_accuracy)

    print(
        f"{col}: accuracy={reduced_accuracy:.4f}, "
        f"absolute difference={differences[col]:.4f}"
    )

print("Smallest difference:", min(differences, key=differences.get))
print("\nQUESTION 6")
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_scores = {}

for c in c_values:
    dv_c = DictVectorizer(sparse=False)
    train_matrix = dv_c.fit_transform(
        df_train.to_dict(orient="records")
    )
    val_matrix = dv_c.transform(
        df_val.to_dict(orient="records")
    )

    model_c = LogisticRegression(
        solver="liblinear", C=c,
        max_iter=1000, random_state=42
    )
    model_c.fit(train_matrix, y_train)
    pred_c = model_c.predict(val_matrix)

    c_scores[c] = accuracy_score(y_val, pred_c)
    print(f"C={c:g}: accuracy={c_scores[c]:.3f}")
best_c = max(c_values, key=lambda c: (c_scores[c], -c))
print("Best C:", best_c)


QUESTION 1
Industry mode: technology

QUESTION 2
interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228
Biggest correlation: ('interaction_count', 'lead_score')

QUESTION 3
lead_source: 0.03
employment_status: 0.02
industry: 0.0
location: 0.0
Biggest MI: lead_source

QUESTION 4
Validation accuracy: 0.65

QUESTION 5
lead_source: accuracy=0.6420, absolute difference=0.0030
number_of_courses_viewed: accuracy=0.6430, absolute difference=0.0020
interaction_count: accuracy=0.6010, absolute difference=0.0440
Smallest difference: number_of_courses_viewed

QUESTION 6
C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
Best C: 0.001
